In [53]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import sys
import os

import cv2
from PIL import Image as PILImage
from rich import print as richprint
import json
import textwrap
from pathlib import Path
import datetime
from collections import Counter
import natsort
from openai import OpenAI
from types import SimpleNamespace

#! #################################################
#!                  ! WARNING !                    # 
#!                                                 #
#!                                                 #
#! USING THIS SCRIPT USES OPENAI CREDITS!!!!!!!!!  #
#!                                                 #
#!                                                 #
#! ##################################################


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
client = OpenAI()

def openai_llm(query, dry_run=False):
    if dry_run:
        return """{
reasoning: The hca (front) is currently visible; to access the internal components for disassembly, we need to turn it over to show the back side.
action: turn,
action_arguments: hca front,
current_module: table_vision,
new_module: table_vision
}"""
    
    completion = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            # {"role": "system", "content": "You are a helpful assistant."},
            {
                "role": "user",
                "content": query
            }
        ]
    )
    msg_content = completion.choices[0].message.content

    return msg_content

In [63]:
def convert_llm_response_to_obj(llm_string):

    output_dict = {}

    lineiterator = iter(llm_string.splitlines())
    for line in lineiterator:
        split_colon = line.split(":")
        if len(split_colon) >= 2:
            key = line.split(":")[0]
            key = key.strip() # remove leading and trailing whitespace

            val = line.split(": ")[1]
            val = val.strip() # remove leading and trailing whitespace
            val = val.rstrip(',') # remove trailing comma
            val = val.strip() # again remove leading and trailing whitespace
            val = val.lower() # lowercase, for now...

            output_dict[key] = val

    output_obj = SimpleNamespace(**output_dict)

    return output_obj

In [68]:
dry_run = True
folder_path = Path("~/vision_pipeline/saves/2024-10-15_13:12:10_knowledge_tree").expanduser()

query_paths = list(folder_path.glob('*_llm_query.txt'))
query_paths = natsort.os_sorted(query_paths)

llm_pred_actions = []
llm_num_correct = 0

for query_path in query_paths:
    richprint("query_path", query_path)
    if query_path.is_file():
        richprint(f"[blue]loading file {query_path}")
        query_file_name = query_path.name
        response_file_name = query_file_name.replace("_llm_query", "_llm_response")
        response_path = query_path.parent / response_file_name

        json_file_name = query_file_name.replace("_llm_query.txt", ".json")
        json_path = query_path.parent / json_file_name

        # check if response file is empty
        if os.stat(response_path).st_size == 0:
            richprint("Response file is empty....")

            with open(query_path, 'r+') as f:
                query_text = f.read()
                richprint("[green]sending query to llm...")
                llm_response = openai_llm(query_text, dry_run=dry_run) #! THIS COSTS MONEY.

            with open(response_path, 'w') as f:
                richprint("writing llm response to file...")
                f.write(llm_response)

            llm_response_obj = None
            if llm_response is not None:
                llm_response_obj = convert_llm_response_to_obj(llm_response)

            richprint("json_path", json_path)
            correct_prediction = None
            llm_pred_action = None
            if llm_response_obj is not None and hasattr(llm_response_obj, 'action') and hasattr(llm_response_obj, 'reasoning'):
                
                richprint("writing predicted action and predicted reasoning to json")

                with open(json_path, 'r') as f:
                    json_data = json.load(f)

                json_data['pred_action'] = llm_response_obj.action
                json_data['pred_reasoning'] = llm_response_obj.reasoning

                llm_pred_action = llm_response_obj.action

                gt_action = json_data['gt_action'] # read from json

                correct_prediction = llm_response_obj.action == gt_action
                json_data["llm_correct_action"] = correct_prediction

                with open(json_path, 'r+') as f:    
                    f.seek(0)
                    json.dump(json_data, f, indent=4)
                    f.truncate()     # remove remaining part


            llm_pred_actions.append(llm_pred_action)
            if correct_prediction:
                llm_num_correct += 1


        else:
            richprint(f"[red] {response_file_name} not empty!")

    break #! DEBUG.

results_path = folder_path / "000_results.json"
with open(results_path, 'r+') as f:
    json_data = json.load(f)
    f.seek(0)
    json_data["llm_pred_actions"] = llm_pred_actions
    json_data["llm_num_correct"] = llm_num_correct
    json.dump(json_data, f, indent=4)
    f.truncate()     # remove remaining part


query_path /home/docker/vision_pipeline/saves/2024-10-15_13:12:10_knowledge_tree/hca_01_1_llm_query.txt

loading file /home/docker/vision_pipeline/saves/2024-10-15_13:12:10_knowledge_tree/hca_01_1_llm_query.txt

Response file is empty....

sending query to llm...

writing llm response to file...

json_path /home/docker/vision_pipeline/saves/2024-10-15_13:12:10_knowledge_tree/hca_01_1.json

writing predicted action and predicted reasoning to json

In [65]:
json_file_name = query_file_name.replace("_llm_query.txt", ".json")
json_path = query_path.parent / json_file_name

#! FIX THIS.






json_path /home/docker/vision_pipeline/saves/2024-10-15_13:12:10_knowledge_tree/hca_01_1.json


In [55]:
# print(llm_response)


response1 = """{
reasoning: The hca (front) is currently visible; to access the internal components for disassembly, we need to turn it over to show the back side.
action: turn,
action_arguments: hca front,
current_module: table_vision,
new_module: table_vision
}"""


response2 = """{
reasoning: "The hca is currently on its front. The next step is to turn the hca over to show the back, so that the components become visible for further disassembly",
action: turn,
action_arguments: hca front,
current_module: table_vision,
new_module: table_vision
}"""


            
output1 = convert_llm_response_to_obj(response1)
print(output1)

# json_llm_response = repair_json(asdf, skip_json_loads=True, return_objects=True)
# # print("good_json_string", good_json_string)

# for key, value, in json_llm_response.items():
    # richprint(f"key: {key}, value: {value}")


# print(fixed_json)



['{']
['reasoning', ' The hca (front) is currently visible; to access the internal components for disassembly, we need to turn it over to show the back side.']
key reasoning
val The hca (front) is currently visible; to access the internal components for disassembly, we need to turn it over to show the back side.
['action', ' turn,']
key action
val turn
['action_arguments', ' hca front,']
key action_arguments
val hca front
['current_module', ' table_vision,']
key current_module
val table_vision
['new_module', ' table_vision']
key new_module
val table_vision
['}']
namespace(action='turn', action_arguments='hca front', current_module='table_vision', new_module='table_vision', reasoning='The hca (front) is currently visible; to access the internal components for disassembly, we need to turn it over to show the back side.')
